[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 模块 2b：玩玩 PyTorch：线性回归

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=960)


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import numpy as np

In [ ]:
torch.__version__

## 热身：用 numpy 做线性回归


我们的模型是：
$$
y_t = 2x^1_t-3x^2_t+1, \quad t\in\{1,\dots,30\}
$$

我们的任务是在给定'观测值' $(x_t,y_t)_{t\in\{1,\dots,30\}}$ 的情况下，恢复出权重 $w^1=2, w^2=-3$ 和偏置 $b = 1$。

为此，我们要解下面这个优化问题：
$$
\underset{w^1,w^2,b}{\operatorname{argmin}} \sum_{t=1}^{30} \left(w^1x^1_t+w^2x^2_t+b-y_t\right)^2
$$

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=1080)


In [ ]:
import numpy as np
from numpy.random import random
# 生成随机输入数据
x = random((30,2))

# 根据输入数据 x 生成对应的标签
y = np.dot(x, [2., -3.]) + 1.
w_source = np.array([2., -3.])
b_source  = np.array([1.])

In [ ]:
x[:5]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

def plot_figs(fig_num, elev, azim, x, y, weights, bias):
    fig = plt.figure(fig_num, figsize=(4, 3))
    plt.clf()
    ax = Axes3D(fig, elev=elev, azim=azim)
    ax.scatter(x[:, 0], x[:, 1], y)
    ax.plot_surface(np.array([[0, 0], [1, 1]]),
                    np.array([[0, 1], [0, 1]]),
                    (np.dot(np.array([[0, 0, 1, 1],
                                          [0, 1, 0, 1]]).T, weights) + bias).reshape((2, 2)),
                    alpha=.5)
    ax.set_xlabel('x_1')
    ax.set_ylabel('x_2')
    ax.set_zlabel('y')
    
def plot_views(x, y, w, b):
    #从不同视角生成不同的图
    elev = 43.5
    azim = -110
    plot_figs(1, elev, azim, x, y, w, b[0])

    plt.show()

In [ ]:
plot_views(x, y, w_source, b_source)

用向量形式表示，我们定义：
$$
\hat{y}_t = {\bf w}^T{\bf x}_t+b
$$
并希望最小化下面的损失：
$$
loss = \sum_t\underbrace{\left(\hat{y}_t-y_t \right)^2}_{loss_t}.
$$

为了最小化损失，我们先计算每个 $loss_t$ 的梯度：
\begin{eqnarray*}
\frac{\partial{loss_t}}{\partial w^1} &=& 2x^1_t\left({\bf w}^T{\bf x}_t+b-y_t \right)\\
\frac{\partial{loss_t}}{\partial w^2} &=& 2x^2_t\left({\bf w}^T{\bf x}_t+b-y_t \right)\\
\frac{\partial{loss_t}}{\partial b} &=& 2\left({\bf w}^T{\bf x}_t+b-y_t \right)
\end{eqnarray*}

注意，损失的真实梯度是：
$$
\frac{\partial{loss}}{\partial w^1} =\sum_t \frac{\partial{loss_t}}{\partial w^1},\quad
\frac{\partial{loss}}{\partial w^2} =\sum_t \frac{\partial{loss_t}}{\partial w^2},\quad
\frac{\partial{loss}}{\partial b} =\sum_t \frac{\partial{loss_t}}{\partial b}
$$

一个 epoch 里，**（批量）梯度下降**按下面的方式更新权重和偏置：
\begin{eqnarray*}
w^1_{new}&=&w^1_{old}-\alpha\frac{\partial{loss}}{\partial w^1} \\
w^2_{new}&=&w^2_{old}-\alpha\frac{\partial{loss}}{\partial w^2} \\
b_{new}&=&b_{old}-\alpha\frac{\partial{loss}}{\partial b},
\end{eqnarray*}

然后跑多个 epoch。


In [ ]:
# 随机初始化可学习的权重和偏置
w_init = random(2)
b_init = random(1)

w = w_init
b = b_init
print("initial values of the parameters:", w, b )

In [ ]:
# 我们的模型前向传播
def forward(x):
    return x.dot(w)+b

# 损失函数
def loss(x, y):
    y_pred = forward(x)
    return (y_pred - y)**2 

print("initial loss:", np.sum([loss(x_val,y_val) for x_val, y_val in zip(x, y)]) )

# 计算梯度
def gradient(x, y):  # d_loss/d_w, d_loss/d_c
    return 2*(x.dot(w)+b - y)*x, 2 * (x.dot(w)+b - y)
 
learning_rate = 1e-2
# 训练循环
for epoch in range(10):
    grad_w = np.array([0,0])
    grad_b = np.array(0)
    l = 0
    for x_val, y_val in zip(x, y):
        grad_w = np.add(grad_w,gradient(x_val, y_val)[0])
        grad_b = np.add(grad_b,gradient(x_val, y_val)[1])
        l += loss(x_val, y_val)
    w = w - learning_rate * grad_w
    b = b - learning_rate * grad_b
    print("progress:", "epoch:", epoch, "loss",l[0])

# 训练结束后
print("estimation of the parameters:", w, b)

In [ ]:
plot_views(x, y, w, b)

## 用张量做线性回归

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=1650)


In [ ]:
dtype = torch.FloatTensor
# # dtype = torch.cuda.FloatTensor # 取消注释即可在 GPU 上运行

In [ ]:
x_t = torch.from_numpy(x).type(dtype)
y_t = torch.from_numpy(y).type(dtype).unsqueeze(1)

这是用张量实现的**（批量）梯度下降**。

注意，主循环里 loss_t 和 gradient_t 这两个函数每次都用相同的输入调用：它们完全可以被合并进循环里（下面我们会这么做）。


In [ ]:
w_init_t = torch.from_numpy(w_init).type(dtype)
b_init_t = torch.from_numpy(b_init).type(dtype)

w_t = w_init_t.clone()
w_t.unsqueeze_(1)
b_t = b_init_t.clone()
b_t.unsqueeze_(1)
print("initial values of the parameters:", w_t, b_t )

In [ ]:
# 我们的模型前向传播
def forward_t(x):
    return x.mm(w_t)+b_t

# 损失函数
def loss_t(x, y):
    y_pred = forward_t(x)
    return (y_pred - y).pow(2).sum()

# 计算梯度
def gradient_t(x, y):  # d_loss/d_w, d_loss/d_c
    return 2*torch.mm(torch.t(x),x.mm(w_t)+b_t - y), 2 * (x.mm(w_t)+b_t - y).sum()

learning_rate = 1e-2
for epoch in range(10):
    l_t = loss_t(x_t,y_t)
    grad_w, grad_b = gradient_t(x_t,y_t)
    w_t = w_t-learning_rate*grad_w
    b_t = b_t-learning_rate*grad_b
    print("progress:", "epoch:", epoch, "loss",l_t)

# 训练结束后
print("estimation of the parameters:", w_t, b_t )

## 用 Autograd 做线性回归

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=1890)


In [ ]:
# 设置 requires_grad=True 表示我们希望反向传播时
# 计算这些张量的梯度。
w_v = w_init_t.clone().unsqueeze(1)
w_v.requires_grad_(True)
b_v = b_init_t.clone().unsqueeze(1)
b_v.requires_grad_(True)
print("initial values of the parameters:", w_v.data, b_v.data )

这是**（批量）梯度下降**的一种实现：不显式地计算梯度，而是使用 autograd。


In [ ]:
for epoch in range(10):
    y_pred = x_t.mm(w_v)+b_v
    loss = (y_pred - y_t).pow(2).sum()
    
    # 用 autograd 计算反向传播。这次调用会计算
    # 损失对所有 requires_grad=True 的变量的梯度。
    # 调用之后，w.grad 和 b.grad 就是保存
    # 损失对 w 和 b 的梯度的张量。
    loss.backward()
    
    # 用梯度下降更新权重。这一步我们只想原地修改
    # w_v 和 b_v 的值，不想为更新步骤构建
    # 计算图，所以用 torch.no_grad() 上下文管理器
    # 阻止 PyTorch 为这些更新构建计算图
    with torch.no_grad():
        w_v -= learning_rate * w_v.grad
        b_v -= learning_rate * b_v.grad
    
    # 更新权重后手动把梯度清零
    # 否则梯度会在每次 .backward() 之后累加
    w_v.grad.zero_()
    b_v.grad.zero_()
    
    print("progress:", "epoch:", epoch, "loss",loss.data.item())

# 训练结束后
print("estimation of the parameters:", w_v.data, b_v.data.t() )

## 用神经网络做线性回归

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=2075)


这是用 nn 包实现的**（批量）梯度下降**。这里的模型超级简单：只有一层，而且没有激活函数！


In [ ]:
# 用 nn 包把我们的模型定义成一系列层。nn.Sequential
# 是一个包含其他 Module 的 Module，它按顺序把输入依次
# 经过这些层产生输出。每个 Linear Module 用线性函数
# 从输入计算输出，并持有内部的权重和偏置变量。
model = torch.nn.Sequential(
    torch.nn.Linear(2, 1),
)

for m in model.children():
    m.weight.data = w_init_t.clone().unsqueeze(0)
    m.bias.data = b_init_t.clone()

# nn 包还包含常用损失函数的定义；这里
# 我们使用均方误差（MSE）作为损失函数。
loss_fn = torch.nn.MSELoss(reduction='sum')

# 切换到训练模式
model.train()

for epoch in range(10):
    # 前向传播：把 x 传给模型计算预测的 y。Module 对象
    # 重载了 __call__ 运算符，所以可以把它们当函数调用。
    # 这样调用时，你把输入数据的 Variable 传给 Module，
    # 它就会产生一个输出数据的 Variable。
    y_pred = model(x_t)
  
    # 注意这个操作等价于：pred = model.forward(x_v)

    # 计算并打印损失。我们传入包含预测值和真实值
    # 的 Variable，损失函数返回一个包含
    # 损失的 Variable。
    loss = loss_fn(y_pred, y_t)

    # 在运行反向传播之前把梯度清零。
    model.zero_grad()

    # 反向传播：计算损失对模型所有可学习参数的梯度。
    # 内部地，每个 Module 的参数都存放在 requires_grad=True 的
    # Variable 里，所以这次调用会为模型中
    # 所有可学习参数计算梯度。
    loss.backward()

    # 用梯度下降更新权重。每个参数都是一个 Tensor，
    # 我们可以像之前一样访问它的数据和梯度。
    with torch.no_grad():
        for param in model.parameters():
            param.data -= learning_rate * param.grad
        
    print("progress:", "epoch:", epoch, "loss",loss.data.item())

# 训练结束后
print("estimation of the parameters:")
for param in model.parameters():
    print(param)

最后一步，我们直接用 optim 包来更新权重和偏置。

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=2390)


In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(2, 1),
)

for m in model.children():
    m.weight.data = w_init_t.clone().unsqueeze(0)
    m.bias.data = b_init_t.clone()

loss_fn = torch.nn.MSELoss(reduction='sum')

model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)


for epoch in range(10):
    y_pred = model(x_t)
    loss = loss_fn(y_pred, y_t)
    print("progress:", "epoch:", epoch, "loss",loss.item())
    # 梯度清零、反向传播、更新权重。
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    
# 训练结束后
print("estimation of the parameters:")
for param in model.parameters():
    print(param)

## 备注

这个问题用 3 行代码就能解！


In [ ]:
xb_t = torch.cat((x_t,torch.ones(30).unsqueeze(1)),1)
# 针对旧版 pytorch
#sol, _ =torch.lstsq(y_t,xb_t)
#sol[:3]
# 适用于 pytorch 1.9 及更新版本
sol = torch.linalg.lstsq(xb_t,y_t)
sol.solution

## 练习：玩玩这段代码


把样本数量从 30 改成 300。会发生什么？怎么修正？


In [ ]:
x = random((300,2))
y = np.dot(x, [2., -3.]) + 1.
x_t = torch.from_numpy(x).type(dtype)
y_t = torch.from_numpy(y).type(dtype).unsqueeze(1)

In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(2, 1),
)

for m in model.children():
    m.weight.data = w_init_t.clone().unsqueeze(0)
    m.bias.data = b_init_t.clone()

loss_fn = torch.nn.MSELoss(reduction = 'sum')

model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)


for epoch in range(10):
    y_pred = model(x_t)
    loss = loss_fn(y_pred, y_t)
    print("progress:", "epoch:", epoch, "loss",loss.item())
    # 梯度清零、反向传播、更新权重。
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    
# 训练结束后
print("estimation of the parameters:")
for param in model.parameters():
    print(param)

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)